# IS 6482 - Week 4 — Ensemble Methods

**Author:** Varun Gupta

**Date:** 1 March, 2026

**Models:**
1. Bagging
2. Random Forest
3. Gradient Boosting
4. AdaBoost
5. (Optional) XGBoost

**Datasets:** Telco customer churn

**Outline:**
- Data load + minimal cleaning
- `Pipeline` + `ColumnTransformer` + `OneHotEncoder` (new!)
- Baseline Decision Tree
- Bagging vs Random Forest (for variance reduction)
- Gradient Boosting (for bias reduction)
- Feature importance: impurity importance, permutation importance


## (0) Import libraries


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt

RANDOM_STATE = 42

## (1) Load the Telco churn dataset


In [ ]:
# If you have a local copy, you can replace this with: df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
telco_url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
df = pd.read_csv(telco_url)

df.shape

In [ ]:
df.head()

## (2) Minimal cleaning

Common quirks in this dataset:
- `customerID` is an ID column → drop it.
- `TotalCharges` loads as text → convert to numeric (coerce errors to NaN).
- Drop rows where `TotalCharges` is NaN
- Separate features (`X`) and labels (`y`)
- `Churn` is Yes/No → convert to 1/0.


In [ ]:
# Drop ID column
if "customerID" in df.columns:
  df = df.drop(columns=["customerID"])

# Convert TotalCharges to numeric (blank strings become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.dropna(inplace=True)  # drop rows with empty TotalCharges

# Target
y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])

# Quick check
y.value_counts(normalize=True).rename("Churn rate")

## (3) Train/test split (stratified)

We stratify so the churn rate is similar in train and test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state = RANDOM_STATE,
    stratify=y
)

X_train.shape, X_test.shape

display(y_train.value_counts(normalize=True))
display(y_test.value_counts(normalize=True))

## (4) Preprocessing with `ColumnTransformer` + `Pipeline` (new concept)

We will setup a Machine Learning pipeline for Data -> Data Transformation -> Model:

Why do this?
- Makes it easy to swap models in/out while keeping preprocessing identical.
- Makes **permutation importance** much easier and more correct.

The Data Transformation we will perform is One-hot encoding of categorical columns.
To prevent **data leakage** the one-hot encoding will be based on training set (that is encoding does not see levels for a categorical feature that are in test but not train).
At test time we will ignore unseen categories.

In practice you can use Data Transformation to also
- Impute numeric columns (e.g. with median)
- Impute categorical columns (e.g., with most-frequent)
- Normalize numeric columns





In [ ]:
# Identify categorical and numeric columns
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

# OneHotEncoder
# (ignore unseen categories at test time)
# Do not produce a sparse output (which require special algorithms)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# ColumnTransformer: OHE for categorical, passthrough for numeric
# verbose_feature_names_out=False gives cleaner names like "gender_Female"
# "num" and "cat" are names that we are defining for the transformations
preprocess = ColumnTransformer(
            transformers=[
                            ("cat", ohe, cat_cols),
                            ("num", "passthrough", num_cols),
                            ],
                    remainder="drop",
                    verbose_feature_names_out=False,
                )

preprocess

## (5) Helper functions (evaluation + feature importance)

We will reuse the same evaluation function across models:
- Accuracy
- F1 (for churn=1)
- AUC
- Confusion matrix
- Classification report


In [ ]:
# Helper function to Evaluate a fitting pipeline
def evaluate_classifier(pipe, X_train, X_test, y_train, y_test, model_name="model"):
    """Fit + evaluate a binary classifier pipeline."""
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (churn=1): {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No churn (0)", "Churn (1)"])
    disp.plot(values_format="d")
    plt.title(f"{model_name} — Confusion Matrix")
    plt.show()

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

    # Optional: ROC-AUC if predict_proba exists
    if hasattr(pipe, "predict_proba"):
        y_proba = pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
        print(f"ROC-AUC: {auc:.4f}")
    else:
        auc = NaN

    return {"model": model_name, "accuracy": acc, "f1": f1, "auc" : auc}

## (6) Baseline model: Decision Tree (what you already know)

This anchors the lesson: ensembles are “many trees combined” to improve performance and stability.


In [ ]:
tree_clf = DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=8,
        )

pipe_tree = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", tree_clf),
        ])

pipe_tree

In [ ]:
pipe_tree.fit(X_train, y_train)
y_pred = pipe_tree.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n=== Decision Tree ===")
print(f"Accuracy: {acc:.4f}")
print(f"F1 (churn=1): {f1:.4f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No churn (0)", "Churn (1)"])
disp.plot(values_format="d")
plt.title("Decision Tree — Confusion Matrix")
plt.show()

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

# Note: ROC-AUC requires the predict_proba function to be available
y_proba = pipe_tree.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.4f}")
result_dict = {"model": "Decision Tree", "accuracy": acc, "f1": f1, "auc" : auc}

results = []
results.append(result_dict)

In [ ]:
# results = []
# results.append(evaluate_classifier(pipe_tree, X_train, X_test, y_train, y_test, model_name="Decision Tree"))

## (7) Bagging (Bootstrap Aggregating)

Idea:
- Fit many trees on bootstrap samples (different “views” of the training data)
- Combine predictions (vote/average)
- **Main benefit:** reduces variance (stabilizes a wiggly model like a tree)


In [ ]:
# BaggingClassifier

bag_model = BaggingClassifier(
                estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                n_estimators=200,
                bootstrap=True,
                oob_score=True,
                n_jobs=-1,
                random_state=RANDOM_STATE,
                )

pipe_bag = Pipeline(steps=[
            ("preprocess", preprocess),
            ("model", bag_model),
            ])
pipe_bag

results.append(evaluate_classifier(pipe_bag, X_train, X_test, y_train, y_test, model_name="Bagging (Trees)"))

# Out-of-bag score (fast-ish internal validation estimate)
print("Bagging OOB score:", pipe_bag.named_steps["model"].oob_score_)

## (8) Random Forest

Random Forest = Bagging + extra randomness:
- still uses bootstrap samples
- **and** at each split, only considers a random subset of features (`max_features` / mtry)

This decorrelates trees → stronger variance reduction.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    oob_score=True,
    max_features="sqrt",   # default in sklearn for classification
)

pipe_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf_model),
])

results.append(evaluate_classifier(pipe_rf, X_train, X_test, y_train, y_test, model_name="Random Forest"))

print("Random Forest OOB score:", pipe_rf.named_steps["model"].oob_score_)

## (9) Gradient Boosting (sequential trees)

Boosting builds trees **sequentially**:
- each new tree tries to correct errors made by the previous ensemble
- **Main benefit:** reduces bias (can fit complex patterns)

Key knobs:
- `n_estimators` = number of trees (stages)
- `learning_rate` = how much to shrink the output of the new tree before adding to ensemble (smaller = slower but often better generalization)


In [ ]:
gb_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
)

pipe_gb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", gb_model),
])

results.append(evaluate_classifier(pipe_gb, X_train, X_test, y_train, y_test, model_name="Gradient Boosting"))

## (10) [OPTIONAL] AdaBoost

AdaBoost is also sequential, but it reweighs examples so the model focuses on “hard-to-classify” points

Here we will use **stumps** (depth=1 trees) as our base classifier.


In [ ]:
ada_model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=300,
            learning_rate=0.5,
            random_state=RANDOM_STATE,
            )

pipe_ada = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", ada_model),
    ])

results.append(evaluate_classifier(pipe_ada, X_train, X_test, y_train, y_test, model_name="AdaBoost (Stumps)"))

## (11) Comparison across the 5 methods

In [ ]:
pd.DataFrame(results).sort_values(by="auc", ascending=False)

## (12) Feature importance

Ensemble methods are not interpretable (unlike shallow decisio trees). We will see **two** approaches for extracting how important each feature for the predictions of the ensemble.

### A) Impurity-based importance / Mean Decrease in Impurity (MDI) -- Fast, already computed while trees are built out, and available directly from tree ensembles
- Available as `feature_importances_` for many tree models
- But can be **biased** toward high-cardinality features and can be misleading when models overfit
- Computed on Training set

### B) Permutation importance (model-agnostic)
- Shuffle one feature column and see how the score drops
- Can be computed on a **held-out** set

We will compute both and discuss differences.


In [ ]:
# Helper function to aggregate importance of one-hot-encoded features based on original categorical column
def aggregate_ohe_importance_to_original(feature_names, importances):
    """Aggregate one-hot feature importances back to the original column name.

    Assumes OHE feature names look like: OriginalCol_Category.
      e.g. OnlineSecurity_No
      because we set verbose_feature_names_out=False
    If we had set verbose_feature_names_out=True our columns name would look like
      cat__OnlineSecurity_Yes, num__TotalCharges
    """
    s = pd.Series(importances, index=feature_names)

    def base_name(name: str) -> str:
        # If the feature is one-hot encoded, take the part before the first underscore

        return name.split("_", 1)[0]

    grouped = s.groupby([base_name(n) for n in s.index]).sum().sort_values(ascending=False)
    return grouped

In [ ]:
# Pick one fitted tree ensemble to inspect (Random Forest is a great default)
pipe_rf.fit(X_train, y_train)

# Feature names after preprocessing (includes one-hot columns)
feat_names = pipe_rf.named_steps["preprocess"].get_feature_names_out()

# Impurity importance
rf_importances = pipe_rf.named_steps["model"].feature_importances_

# This is the importance of the transformed (one-hot encoded features)
# Plot a horizontal bar chart
top_n = 25 # How many features to plot
imp = pd.Series(rf_importances, index=feat_names).sort_values(ascending=False).head(top_n)[::-1]
plt.figure(figsize=(8, max(4, 0.25 * top_n)))
plt.barh(imp.index, imp.values)
plt.title(f"Random Forest — impurity-based feature importance (top {top_n})")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Aggregate one-hot importances back to original columns
rf_grouped = aggregate_ohe_importance_to_original(feat_names, rf_importances)
print("="*70)
print("One-hot importances aggregated back to original columns:")
print("="*70)
rf_grouped.head(15)

In [ ]:
# Permutation importance on the *pipeline*:
# - shuffles ORIGINAL columns in X_test
# - measures drop in score for this fitted pipeline
#
# We'll use F1 because it's in the classification report and focuses on the positive class.
pipe_rf.fit(X_train, y_train)

r = permutation_importance(
    pipe_rf,
    X_test,
    y_test,
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="f1",
)

perm_series = pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False)
perm_series.head(15)

## (13) [OPTIONAL] Choosing `max_features` (mtry) for Random Forest

Two common settings for classification:
- `sqrt` (default in sklearn)
- `log2`

We’ll do a **small** CV experiment (3-fold) to show how this knob can matter.


In [ ]:
# BONUS: quick demo (small grid, small CV)

candidates = ["sqrt", "log2", 1.0]  # 1.0 = use all features at each split (closer to pure bagging)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

scores = []
for mf in candidates:
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_features=mf,
        )),
    ])
    cv_score = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()
    scores.append(cv_score)

pd.DataFrame({"max_features": candidates, "cv_f1": scores}).sort_values("cv_f1", ascending=False)

## (14) [OPTIONAL] Choosing number of trees in boosting (`n_estimators`)

Boosting is sequential. Let us see how the performance changes as each tree was added to the ensemble. We will:
- Fit one big boosting model
- Track performance **stage by stage**

We will plot F1 on the test set by boosting stage.
In “real life” we would do this on a validation set, not the test set.


In [ ]:
# BONUS: stage-by-stage curve for Gradient Boosting

pipe_gb.fit(X_train, y_train)

# Transform once, then use staged_predict on the underlying model
X_test_trans = pipe_gb.named_steps["preprocess"].transform(X_test)
gb_est = pipe_gb.named_steps["model"]

f1_by_stage = []
for y_pred_stage in gb_est.staged_predict(X_test_trans):
    f1_by_stage.append(f1_score(y_test, y_pred_stage))

best_stage = int(np.argmax(f1_by_stage)) + 1  # stages are 1-indexed for humans
print("Best stage (by test F1):", best_stage, "out of", len(f1_by_stage))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(f1_by_stage) + 1), f1_by_stage)
plt.axvline(best_stage, linestyle="--")
plt.xlabel("Number of trees (stages)")
plt.ylabel("F1 (test)")
plt.title("Gradient Boosting — F1 by stage (demo)")
plt.tight_layout()
plt.show()

## Wrap-up questions

1. Which methods seemed to help the most: bagging/RF or boosting?
2. Which metric should we care about most for churn: accuracy, recall, F1, ROC-AUC?
3. Why might impurity-based importance disagree with permutation importance?
4. If two features are strongly correlated, what happens to permutation importance?
